# Bonus Analysis

In this notebook I wanted to explore some extra angles on the Tallahassee crime data — specifically looking at time-of-day patterns, weekend vs weekday differences, and which areas tend to have more serious incidents. I also built a small helper module (`functions.py`) so I could reuse plotting code across notebooks.

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from functions import plot_hist, plot_bar, plot_scatter, summary_by_group, top_n_by_group

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/processed/tops_crime_data_cleaned.csv')
df['CREATE_TIME_INCIDENT'] = pd.to_datetime(df['CREATE_TIME_INCIDENT'], errors='coerce')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## Custom Feature Engineering

I added a few new columns to make the analysis more interesting — things like what time of day an incident happened, whether it was a weekend, and a rough severity score based on the incident type.

In [ ]:
# extract hour and day from timestamp
df['hour'] = df['CREATE_TIME_INCIDENT'].dt.hour
df['day_of_week'] = df['CREATE_TIME_INCIDENT'].dt.day_name()

# true if saturday or sunday
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# bucket hours into time of day categories
def categorize_time(hour):
    if 6 <= hour < 12:
        return 'Morning (6-11)'
    elif 12 <= hour < 17:
        return 'Afternoon (12-16)'
    elif 17 <= hour < 21:
        return 'Evening (17-20)'
    else:
        return 'Night (21-5)'

df['time_of_day'] = df['hour'].apply(categorize_time)

# give each incident type a severity score (1=low, 2=medium, 3=high)
# low = minor stuff like noise complaints, high = violent crimes
severity_map = {
    'MISC SERVICE CALL': 1, 'COMMUNITY POLICING': 1, '911 HANGUP': 1,
    'LOUD NOISE / MUSIC': 1, 'FOUND PROPERTY': 1, 'ANIMAL COMPLAINT': 1,
    'MOTORIST ASSIST': 1, 'PARKING COMPLAINT': 1, 'ALARM': 1,
    'WELFARE CHECK': 1, 'CIVIL COMPLAINT': 1,
    'SUSPICIOUS': 2, 'CRASH W/O INJURIES': 2, 'TRESPASSING': 2,
    'TRESPASS WARNING': 2, 'WANTED PERSON': 2, 'TRAFFIC STOP': 2,
    'DISTURBANCE': 2, 'MENTAL HEALTH': 2, 'DRUG': 2, 'WARRANT': 2,
    'DISORDERLY': 2, 'FRAUD': 2, 'THEFT': 2,
    'BATTERY': 3, 'ROBBERY': 3, 'ASSAULT': 3, 'SHOOTING': 3,
    'HOMICIDE': 3, 'BURGLARY': 3, 'STOLEN VEHICLE': 3,
    'ARMED': 3, 'DOMESTIC': 3, 'KIDNAPPING': 3, 'CRASH WITH INJURIES': 3,
}
df['severity_score'] = df['DISPO_TEXT'].map(severity_map).fillna(2).astype(int)
df['severity_label'] = df['severity_score'].map({1: 'Low', 2: 'Medium', 3: 'High'})

print("New features added successfully!")
print(f"\ntime_of_day value counts:\n{df['time_of_day'].value_counts()}")
print(f"\nseverity_score distribution:\n{df['severity_score'].value_counts().sort_index()}")
print(f"\nis_weekend distribution:\n{df['is_weekend'].value_counts()}")

df[['CREATE_TIME_INCIDENT', 'DISPO_TEXT', 'hour', 'day_of_week',
    'is_weekend', 'time_of_day', 'severity_score', 'severity_label']].head(8)

## Time-of-Day Analysis

Which part of the day sees the most incidents? I bucketed hours into Morning, Afternoon, Evening, and Night to see if there's a clear pattern.

In [ ]:
tod_order = ['Morning (6-11)', 'Afternoon (12-16)', 'Evening (17-20)', 'Night (21-5)']
tod_counts = df['time_of_day'].value_counts().reindex(tod_order).reset_index()
tod_counts.columns = ['time_of_day', 'count']

# get avg severity per time of day using our helper
tod_stats = summary_by_group(df, 'time_of_day', 'severity_score',
                              agg_funcs=('count', 'mean'))
tod_stats.columns = ['Time of Day', 'Incident Count', 'Avg Severity']
print("=== Incidents and Avg Severity by Time of Day ===")
print(tod_stats.to_string(index=False))

colors = ['#1a3a5c', '#2166ac', '#4393c3', '#92c5de']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(tod_counts['time_of_day'], tod_counts['count'],
              color=colors, edgecolor='white', linewidth=0.5)

# add count labels on top of each bar
for bar, val in zip(bars, tod_counts['count']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Incidents by Time of Day — Tallahassee 2025–2026',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Time of Day', fontsize=12)
ax.set_ylabel('Number of Incidents', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('../figures/time_of_day_incidents.png', dpi=300)
plt.show()

Night (9 PM – 5 AM) actually has the most incidents overall, which was surprising at first — but it makes sense when you consider that window covers 8 hours vs the other buckets covering 4-5. Afternoon is the busiest on a per-hour basis. Severity stays pretty consistent across all time periods, so serious incidents aren't concentrated in any one part of the day.

## Weekend vs Weekday Patterns

Do crime patterns change on weekends? I normalized by number of days (5 weekdays, 2 weekend days) to make it a fair comparison.

In [ ]:
weekend_totals = df.groupby('is_weekend').size().reset_index(name='total')
weekend_totals['label'] = weekend_totals['is_weekend'].map({False: 'Weekday', True: 'Weekend'})
# normalize by number of days so the comparison is fair
weekend_totals['days_in_period'] = weekend_totals['is_weekend'].map({False: 5, True: 2})
weekend_totals['avg_per_day'] = (weekend_totals['total'] / weekend_totals['days_in_period']).round(0).astype(int)
print("=== Weekday vs Weekend Summary ===")
print(weekend_totals[['label', 'total', 'avg_per_day']].rename(
    columns={'label': 'Period', 'total': 'Total Incidents', 'avg_per_day': 'Avg Per Day'}).to_string(index=False))

top_types = df['DISPO_TEXT'].value_counts().head(5).index.tolist()
wk_breakdown = (
    df[df['DISPO_TEXT'].isin(top_types)]
    .groupby(['DISPO_TEXT', 'is_weekend'])
    .size()
    .reset_index(name='count')
)
wk_breakdown['period'] = wk_breakdown['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

weekday_vals = (wk_breakdown[wk_breakdown['period'] == 'Weekday']
                .set_index('DISPO_TEXT')['count'].reindex(top_types).fillna(0))
weekend_vals = (wk_breakdown[wk_breakdown['period'] == 'Weekend']
                .set_index('DISPO_TEXT')['count'].reindex(top_types).fillna(0))

x = np.arange(len(top_types))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width / 2, weekday_vals, width, label='Weekday', color='#2166ac', edgecolor='white')
ax.bar(x + width / 2, weekend_vals, width, label='Weekend', color='#f4a582', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([t[:28] for t in top_types], rotation=15, ha='right', fontsize=10)
ax.set_title('Top 5 Incident Types: Weekday vs Weekend', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Incidents', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('../figures/weekday_vs_weekend_top_crimes.png', dpi=300)
plt.show()

Per day, weekends are busier than weekdays. Community Policing calls drop a lot on weekends which makes sense — that's more of a proactive patrol activity. Noise complaints and suspicious calls go up, which fits with more people being out on Friday and Saturday nights.

## Incident Heatmap — Hour × Day of Week

I wanted to see if certain days AND hours combined were consistently busier — a heatmap felt like the right way to show both dimensions at once instead of doing two separate charts.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# pivot table: rows = hour, columns = day of week, values = incident count
heat_pivot = df.pivot_table(
    index='hour',
    columns='day_of_week',
    values='DISPO_TEXT',
    aggfunc='count'
)[day_order]

# find the busiest hour/day combo for annotation
peak_idx = heat_pivot.stack().idxmax()
peak_hour, peak_day = peak_idx
peak_val = int(heat_pivot.loc[peak_hour, peak_day])
peak_col_idx = day_order.index(peak_day)

fig, ax = plt.subplots(figsize=(13, 9))

sns.heatmap(
    heat_pivot,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Number of Incidents', 'shrink': 0.75},
)

# highlight the peak cell
ax.add_patch(plt.Rectangle(
    (peak_col_idx, peak_hour), 1, 1,
    fill=False, edgecolor='black', lw=2.5
))

# label the peak with an arrow
ax.annotate(
    f'Peak: {peak_val:,} incidents\n{peak_day}, {peak_hour:02d}:00',
    xy=(peak_col_idx + 0.5, peak_hour + 0.5),
    xytext=(peak_col_idx + 3.0, peak_hour - 4.5),
    fontsize=10, fontweight='bold', color='black',
    arrowprops=dict(arrowstyle='->', color='black', lw=1.8),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='grey', alpha=0.85),
)

ax.set_title(
    'Incident Frequency by Hour of Day and Day of Week\nTallahassee TOPS Data 2025–2026',
    fontsize=15, fontweight='bold', pad=14
)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Hour of Day (24h clock)', fontsize=12)
ax.set_yticklabels([f'{h:02d}:00' for h in range(24)], rotation=0, fontsize=8)
ax.set_xticklabels(day_order, rotation=30, ha='right', fontsize=10)

fig.tight_layout()
fig.savefig('../figures/hour_dayofweek_heatmap.png', dpi=300)
plt.show()
print(f"Peak cell: {peak_day} at {peak_hour:02d}:00 with {peak_val:,} incidents")

The afternoon band (roughly 10 AM – 6 PM) is consistently the hottest across every day. Friday and Saturday nights stand out compared to other weeknights. Early mornings (2–6 AM) are the quietest no matter what day it is.

## Severity Score by Location

Which areas in Tallahassee tend to have the most serious incidents? I only looked at locations with at least 100 incidents so the averages are meaningful.

In [ ]:
sev_summary = summary_by_group(df, 'severity_label', 'hour', agg_funcs=('count', 'mean'))
sev_summary.columns = ['Severity', 'Total Incidents', 'Avg Hour of Day']
print("=== Overall Incident Count by Severity Level ===")
print(sev_summary.to_string(index=False))

# only keep locations with at least 100 incidents so averages are meaningful
area_sev = (
    df.dropna(subset=['LOCATION_TEXT'])
    .groupby('LOCATION_TEXT')
    .agg(total=('severity_score', 'count'),
         avg_severity=('severity_score', 'mean'))
    .query('total >= 100')
    .sort_values('avg_severity', ascending=False)
    .head(15)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('RdYlGn_r', len(area_sev))
ax.barh(area_sev['LOCATION_TEXT'], area_sev['avg_severity'],
        color=colors, edgecolor='white', linewidth=0.4)

# reference lines to show where low/medium/high boundaries are
ax.axvline(1.5, color='#4575b4', linestyle='--', linewidth=1.2, alpha=0.8, label='Low / Medium boundary')
ax.axvline(2.5, color='#d73027', linestyle='--', linewidth=1.2, alpha=0.8, label='Medium / High boundary')

ax.set_title('Top 15 Locations by Average Incident Severity Score\n(locations with \u2265 100 incidents)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Average Severity Score  (1 = Low, 2 = Medium, 3 = High)', fontsize=11)
ax.set_ylabel('Location', fontsize=11)
ax.legend(fontsize=9, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('../figures/severity_by_area.png', dpi=300)
plt.show()

## Using the Helper Functions

Here I'm using the functions from `functions.py` to avoid rewriting the same plotting code over and over.

In [ ]:
# histogram of incidents by hour
fig, ax = plot_hist(
    df, col='hour',
    title='Distribution of Incidents by Hour of Day',
    xlabel='Hour (24h clock)',
    bins=24, color='#2166ac',
    save_path='../figures/hour_distribution.png'
)
plt.show()

# grouped stats by day of week
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_summary = summary_by_group(df, 'day_of_week', 'severity_score',
                                agg_funcs=('count', 'mean', 'std'))
dow_summary.columns = ['Day of Week', 'Total Incidents', 'Avg Severity', 'Std Severity']
dow_summary['Day of Week'] = pd.Categorical(dow_summary['Day of Week'], categories=dow_order, ordered=True)
dow_summary = dow_summary.sort_values('Day of Week')
print("\n=== Incident Summary by Day of Week ===")
print(dow_summary.to_string(index=False))

# top 10 incident types
top10 = top_n_by_group(df, 'DISPO_TEXT', n=10)
fig, ax = plot_bar(
    top10, x_col='DISPO_TEXT', y_col='count',
    title='Top 10 Incident Types',
    xlabel='Incident Type', ylabel='Number of Incidents',
    horizontal=True, figsize=(10, 6),
    save_path='../figures/top10_via_helper.png'
)
plt.show()

## Summary

A few things stood out from this analysis:

- Night has the most raw incidents but that's partly because it's the longest time window. Afternoon is the peak on a per-hour basis.
- Weekends are busier per day than weekdays, with a noticeable shift in the type of incidents.
- The heatmap made it clear that Friday and Saturday evenings are the combined hotspot — something you wouldn't easily see from a line plot or bar chart alone.
- The severity score showed which specific locations tend to attract more serious calls, which could be useful for resource allocation.